# K_01c – Animierte Energiefluss-Karten

**Grid-Arbitrage** · Animierte Visualisierung: Erzeugung, Verbrauch, Zonenfluss, Cross-Border

**Gruppe:** SC26_Gruppe_2 | **Status:** Kür – explorativ | **Abhängig von:** K_01 (Cache), K_01b (Zonen)

---

> 🎬 **5 GIF-Typen × 2 Zeitskalen = 10 Animationen**
>
> | GIF | Inhalt | Tagesverlauf | Jahresverlauf |
> |-----|--------|:---:|:---:|
> | A | Erzeugungsschwankungen (Dots nach ET, Grösse = MW) | 24h × 4 Jahreszeiten | 52 Wochen |
> | B | Verbrauchsschwankungen (Dots nach Zone, Grösse = Last) | 24h × 4 Jahreszeiten | 52 Wochen |
> | C | Zonenfluss (Moving Dots auf Pfaden, Dichte = MW) | 24h | 52 Wochen |
> | D | Cross-Border Import/Export (Dots an Grenzen) | 24h | 52 Wochen |
> | E | Kombiniert C + D | 24h | 52 Wochen |
>
> ⚠️ **Alle Flussdaten sind modelliert (synthetisch)** — keine Swissgrid-Messdaten.
> Vollständige Warnung in Sektion 2.

*Voraussetzung: K_01 ausgeführt (BFE/BFS Cache). K_01b für Zonenfarben (optional — Fallback vorhanden).*


| [← K_01b Zonenmodell](K_01b_Zonenmodell_Erweitert.ipynb) | [↑ Übersicht](../organisation/O_01_Project_Overview.ipynb) | [K_02 Cross-Border →](K_02_Cross_Border.ipynb) |
|:---|:---:|---:|

## Inhaltsverzeichnis<a id='toc_K_01c'></a>

[Einleitung](#einleitung_K_01c)  
[Initialisierung](#init_K_01c)  
[Warnung: Synthetische Flussdaten](#warnung_K_01c)  
1 [Geodaten & Zonengeometrie](#geodaten_K_01c)  
2 [Datenprofil-Modell](#profile_K_01c)  
3 [Helper-Funktionen](#helper_K_01c)  
4 [GIF A — Erzeugungsschwankungen](#gif_a_K_01c)  
5 [GIF B — Verbrauchsschwankungen](#gif_b_K_01c)  
6 [GIF C — Zonenfluss](#gif_c_K_01c)  
7 [GIF D — Cross-Border Import/Export](#gif_d_K_01c)  
8 [GIF E — Kombiniert (Zonenfluss + Cross-Border)](#gif_e_K_01c)  
[Abschluss](#abschluss_K_01c)  


---
## Einleitung<a id='einleitung_K_01c'></a>

[↑ Inhaltsverzeichnis](#toc_K_01c)

**Animierte Energiefluss-Karten** für die Schweizer Zonen — Rendering von GIFs
aus synthetischen Zeitverläufen, aufgebaut aus realen BFE-Zonenbilanzen:

- **GIF A** — Erzeugungsschwankungen je Zone
- **GIF B** — Verbrauchsschwankungen je Zone
- **GIF C** — Inner-CH-Zonenfluss (Netto-Energieflüsse zwischen Zonen)
- **GIF D** — Cross-Border Import/Export
- **GIF E** — Kombinierte Darstellung (C + D)

> Die Flussdaten sind **synthetisch** (Modellierung aus Zonenbilanzen),
> nicht aus realen Messdaten. Details siehe Warnung im nächsten Abschnitt.


---
## Initialisierung<a id='init_K_01c'></a>

[↑ Inhaltsverzeichnis](#toc_K_01c)

In [ ]:
# ── lib/ aus Projekt-Root erreichbar machen + lib-Imports ───────────────────
# Notebook liegt in einem Unterordner (kuer/, experimental/, notebooks/,
# organisation/). Damit 'from lib.xxx import ...' funktioniert, muss der
# Projekt-Root vorne in sys.path stehen. autoreload sorgt dafür, dass
# Änderungen in lib/*.py ohne Kernel-Restart übernommen werden.
import sys, os
_PROJECT_ROOT = os.path.abspath('..')
if _PROJECT_ROOT not in sys.path:
    sys.path.insert(0, _PROJECT_ROOT)
try:
    get_ipython().run_line_magic('load_ext', 'autoreload')
    get_ipython().run_line_magic('autoreload', '2')
except Exception:
    pass

# lib-Imports (einmal zentral — in allen folgenden Zellen verfügbar)
from lib.widgets  import show_animation
from lib.plotting import show_source, should_skip

print(f'lib-Pfad aktiv: {_PROJECT_ROOT}/lib')


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# ⚙ PFAD-KONFIGURATION — experimental/ Notebook
# ══════════════════════════════════════════════════════════════════════════════
import os

EXP_BASE       = '.'
EXP_DATA_DIR   = os.path.join(EXP_BASE, 'data', 'raw')
EXP_INTER_DIR  = os.path.join(EXP_BASE, 'data', 'intermediate')
EXP_OUTPUT_DIR = os.path.join(EXP_BASE, 'output', 'charts')

PROD_BASE      = '..'
PROD_DATA_DIR  = os.path.join(PROD_BASE, 'data', 'raw')
PROD_INTER_DIR = os.path.join(PROD_BASE, 'data', 'intermediate')
PROD_SYNC_DIR  = os.path.join(PROD_BASE, 'sync')

for d in [EXP_DATA_DIR, EXP_INTER_DIR, EXP_OUTPUT_DIR]:
    os.makedirs(d, exist_ok=True)

# ── Granularität ─────────────────────────────────────────────────────────────
GRANULARITY = 'kantone'   # ⚙  'kantone' / 'bezirke' / 'gemeinden' / 'nuts3'
CC_CODE     = 'CH'        # ⚙  ISO 3166-1 Alpha-2
ENTSOE_BZ   = '10YCH-SWISSGRIDZ'  # ⚙  Für spätere ENTSO-E-Einspeisung

# ── Animation-Parameter ──────────────────────────────────────────────────────
# Tagesanimation: 24h × FRAMES_PER_HOUR = Total-Frames
#   10 Frames/Stunde → 240 Frames gesamt
# Jahresanimation: 52 Wochen × FRAMES_PER_WEEK = Total-Frames
#   5 Frames/Woche → 260 Frames gesamt
# Alle Zwischenwerte werden cubic-interpoliert (CubicSpline).

FRAMES_PER_HOUR = 2   # ⚙ Frames je Stunde  (24h × 4 = 96 Frames)
FRAMES_PER_WEEK = 2   # ⚙ Frames je Woche   (52W × 4 = 208 Frames)

N_FRAMES_TAG  = 24 * FRAMES_PER_HOUR   # = 96
N_FRAMES_JAHR = 52 * FRAMES_PER_WEEK   # = 208

FPS_TAG       = 10   # ⚙ fps → 96/10 = 9.6s Loop
FPS_JAHR      = 10   # ⚙ fps → 208/10 = 20.8s Loop
# DPI_GIF       = 130  # ⚙
DPI_GIF       = 90  # ⚙

print(f"⚙ Pfade:")
print(f"  EXP_OUTPUT_DIR = {os.path.abspath(EXP_OUTPUT_DIR)}")
print(f"  PROD_DATA_DIR  = {os.path.abspath(PROD_DATA_DIR)}")
print(f"  Granularität   = {GRANULARITY} | Land = {CC_CODE}")
print(f"  Frames: Tag={N_FRAMES_TAG}f ({FRAMES_PER_HOUR}f/h) @{FPS_TAG}fps = {N_FRAMES_TAG/FPS_TAG:.0f}s Loop")
print(f"  Frames: Jahr={N_FRAMES_JAHR}f ({FRAMES_PER_WEEK}f/Woche) @{FPS_JAHR}fps = {N_FRAMES_JAHR/FPS_JAHR:.0f}s Loop")


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# ⚙ ANIMATIONS-SCHALTER — hier alle visuellen Parameter einstellen
# Änderungen hier wirken auf alle GIF A/C/D/E Zellen
# ══════════════════════════════════════════════════════════════════════════════

# ── GIF A: Erzeugungsdots ─────────────────────────────────────────────────────
GIF_A_CLUSTER_DEG  = 0.15   # ⚙ Cluster-Radius [Grad] (~13 km)
                              #   klein (0.08) = mehr Einzelpunkte
                              #   gross (0.25) = stärker zusammengefasst
GIF_A_SIZE_SCALE   = 160    # ⚙ Dot-Grösse Basis
                              #   Grösse = sqrt(mw / et_max) × GIF_A_SIZE_SCALE
GIF_A_ALPHA_MIN    = 0.0    # ⚙ Deckkraft bei CF=0 (Solar Nacht: 0.0 = unsichtbar)
GIF_A_ALPHA_MAX    = 0.92   # ⚙ Deckkraft bei CF=1 (Volllast)

# ── GIF C/D/E: Fluss-Dots ────────────────────────────────────────────────────
GIF_C_NORM_MODE      = 'per_path'  # ⚙ 'global'   = alle Pfade relativ zu GIF_C_MW_NORM_GLOBAL
                                    #    'per_path' = jeder Pfad relativ zu seinem eigenen Ø
GIF_C_MW_NORM_GLOBAL = 2500        # ⚙ Normierungswert für 'global' [MW]
GIF_C_N_DOTS_BASE    = 5           # ⚙ Anzahl Dots bei Vollauslastung
GIF_C_DOT_SIZE       = 45          # ⚙ Dot-Grösse [pt²] — fix, Anzahl variiert
GIF_C_SHOW_MW_LABEL  = True        # ⚙ MW-Wert auf Pfaden anzeigen

# Zusammenfassung
print("⚙ Animations-Schalter:")
print(f"  GIF A: cluster={GIF_A_CLUSTER_DEG}° | size_scale={GIF_A_SIZE_SCALE} | "
      f"alpha {GIF_A_ALPHA_MIN:.2f}->{GIF_A_ALPHA_MAX:.2f}")
print(f"  GIF C/D/E: norm='{GIF_C_NORM_MODE}' | n_dots_base={GIF_C_N_DOTS_BASE} | "
      f"dot_size={GIF_C_DOT_SIZE}")


---
### ⚙ Animations-Schalter — alle in einer Zelle einstellbar

| Schalter | Default | Beschreibung |
|----------|---------|--------------|
| `GIF_A_CLUSTER_DEG` | `0.15` | Cluster-Radius [Grad, ~13 km]. Kleiner = mehr Punkte, Grösser = weniger |
| `GIF_A_SIZE_SCALE` | `160` | Dot-Basisgrösse. Effektiv: `sqrt(mw/et_max) × scale` |
| `GIF_A_ALPHA_MIN/MAX` | `0.0/0.92` | Deckkraft bei CF=0 / CF=1. Solar bei Nacht: `0.0` = unsichtbar |
| `GIF_C_NORM_MODE` | `'per_path'` | `'global'`: alle Pfade an `GIF_C_MW_NORM_GLOBAL` normiert. `'per_path'`: je Pfad an eigenem Ø |
| `GIF_C_MW_NORM_GLOBAL` | `2500` | Normierungswert für `'global'` [MW] |
| `GIF_C_N_DOTS_BASE` | `5` | Dots bei Vollauslastung. Skaliert linear mit `|flow/norm|` |
| `GIF_C_DOT_SIZE` | `45` | Dot-Grösse fix [pt²]. Nur Anzahl variiert, nicht Grösse |
| `GIF_C_SHOW_MW_LABEL` | `True` | MW-Wert auf Pfaden anzeigen |


In [ ]:
# ── Bibliotheken ─────────────────────────────────────────────────────────────
import os, warnings, json
import subprocess, sys
from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors
import matplotlib.ticker as mticker
from matplotlib.colors import Normalize
from matplotlib import cm
from matplotlib.animation import FuncAnimation, PillowWriter
import requests
warnings.filterwarnings('ignore')

for pkg in [('geopandas','geopandas'),('scipy','scipy'),('pillow','PIL')]:
    try: __import__(pkg[1])
    except ImportError: subprocess.check_call([sys.executable,'-m','pip','install',pkg[0],'-q'])

import geopandas as gpd

print(f"Pandas     : {pd.__version__}")
print(f"GeoPandas  : {gpd.__version__}")
print(f"NumPy      : {np.__version__}")
print(f"Matplotlib : {plt.matplotlib.__version__}")
print(f"📅 Ausgeführt: {datetime.now().strftime('%d.%m.%Y um %H:%M:%S')}")


In [ ]:
# ── Config + Farben (SSOT — nur lesend) ─────────────────────────────────────
with open('../sync/config.json') as _f: CFG = json.load(_f)

SZ_AKTIV   = CFG['szenarien']['gleichzeitigkeit_aktiv']
CHARTS_DIR = EXP_OUTPUT_DIR
DATA_DIR   = PROD_DATA_DIR
INTER_DIR  = PROD_INTER_DIR
EXP_CHARTS_DIR = EXP_OUTPUT_DIR

DPI_GIF    = 130   # ⚙ Animationsauflösung (höher = grössere Dateien)
FPS_TAG    = 6     # ⚙ Frames/Sekunde Tagesverlauf (24 Frames → ~4s Loop)
FPS_JAHR   = 8     # ⚙ Frames/Sekunde Jahresverlauf (52 Frames → ~6.5s Loop)

_viz         = CFG.get('visualisierung', {}).get('farben', {})
BG_DARK      = _viz.get('bg_dark',    '#0d1117')
BG_PANEL     = _viz.get('bg_panel',   '#141414')
C_LOAD       = _viz.get('c_load',     '#66BB6A')
C_CHARGE     = _viz.get('c_charge',   '#1565C0')
C_FEED       = _viz.get('c_feed',     '#B71C1C')
C_PRICE      = _viz.get('c_price',    '#FFA726')
C_SPINE      = _viz.get('c_spine',    '#333333')
C_ACHSE      = _viz.get('c_achse',    '#aaaaaa')
C_TICK       = _viz.get('c_tick',     '#bbbbbb')
C_LEGENDE_BG = _viz.get('c_legende_bg','#111111')
C_SOLAR      = _viz.get('c_solar',    '#FDD835')
C_GRENZWERT  = _viz.get('c_amber_dark','#FF6F00')
C_GRUEN_DARK = _viz.get('c_gruen_dark','#388E3C')
C_CYAN       = _viz.get('c_cyan',     '#26C6DA')
_stil        = CFG.get('visualisierung',{}).get('stil',{})
FS_TITEL     = _stil.get('schriftgroesse_titel', 13)
FS_KLEIN     = _stil.get('schriftgroesse_klein',  7)

# K_01b Zonenfarben (Fallback wenn K_01b nicht ausgeführt)
ZONE_COLORS_B = {
    'Nordost'        : '#1565C0',
    'Ostschweiz'     : '#00838F',
    'Mitte-Erzeugung': '#7B1FA2',
    'Mitte-Transit'  : '#388E3C',
    'West'           : '#FF6F00',
    'Süd'            : '#B71C1C',
}
ET_COLORS = {
    'Solar':      C_SOLAR,      'Wasserkraft': C_CHARGE,
    'Kernkraft':  '#7B1FA2',    'Wind':        C_CYAN,
    'Biomasse':   C_GRUEN_DARK, 'Andere':      '#9E9E9E',
}
print(f"Config geladen | Szenario: {SZ_AKTIV} | Charts: {CHARTS_DIR}")


---
## Warnung: Synthetische Flussdaten<a id='warnung_K_01c'></a>

[↑ Inhaltsverzeichnis](#toc_K_01c)

---

> ## ⚠️ ALLE ENERGIEFLÜSSE IN DIESEM NOTEBOOK SIND MODELLIERT — KEINE MESSDATEN
>
> ### Was ist synthetisch:
> - **Zonenflüsse** (GIF C/E): Aus Imbalance-Differenzen abgeleitet, kein echter Lastfluss-Solver
> - **Stündliche Profile**: Gauss-/Sinus-Kurven nach Energieträgertypik — keine ENTSO-E Stundendaten
> - **Jahresverlauf**: Saisonale Kapazitätsfaktoren (BFE 2023) × synthetisches Wochenprofil
> - **Cross-Border** (GIF D/E): Netto-Imbalance der CH-Zonen als Proxy für Grenzflüsse
>
> ### Was real ist:
> - **Installierte Kapazitäten** pro Zone aus BFE GeoPackage (322k Anlagen)
> - **Zonenstruktur** aus K_01b (6-Zonen-Modell)
> - **Kapazitätsfaktoren** (BFE Elektrizitätsstatistik 2023)
> - **Geographische Positionen** (Zone-Zentroide aus BFS Kantonsdaten)
>
> ### Zweck dieser Animationen:
> Strukturelle Muster sichtbar machen (Duck Curve, Winter-Peak, Süd→Nord-Fluss),
> **nicht** operative Lastfluss-Simulation. Für Investitionsentscheide:
> Swissgrid SCADA-Daten oder professionellen Netzplaner hinzuziehen.

---


---
## 1. Geodaten & Zonengeometrie<a id='geodaten_K_01c'></a>

[↑ Inhaltsverzeichnis](#toc_K_01c)

In [ ]:
# ── Kantonsgrenzen laden (K_01-Cache: kantone.gpkg) ──────────────────────────
KANT_CANDIDATES = [
    os.path.join(PROD_DATA_DIR, 'kantone.gpkg'),  # Aus K_01-Cache              # K_01 Standard-Output
    os.path.join(PROD_DATA_DIR, 'swissboundaries3d.gpkg'),
    os.path.join(DATA_DIR, 'swissBOUNDARIES3D_1_5_TLM_KANTONSGEBIET.gpkg'),
    os.path.join('..', 'data', 'raw', 'kantone.gpkg'),
]
KANT_NUM_TO_ABK = {
    1:'ZH',2:'BE',3:'LU',4:'UR',5:'SZ',6:'OW',7:'NW',8:'GL',
    9:'ZG',10:'FR',11:'SO',12:'BS',13:'BL',14:'SH',15:'AR',16:'AI',
    17:'SG',18:'GR',19:'AG',20:'TG',21:'TI',22:'VD',23:'VS',24:'NE',25:'GE',26:'JU',
}

gdf_kant = None
for path in KANT_CANDIDATES:
    if os.path.exists(path) and os.path.getsize(path) > 50_000:
        try:
            layers = gpd.list_layers(path)
            lname = next((l for l in layers['name'] if 'kanton' in l.lower()),
                         layers['name'].iloc[0])
            gdf_raw = gpd.read_file(path, layer=lname)
            if gdf_raw.crs and gdf_raw.crs.to_epsg() != 4326:
                gdf_raw = gdf_raw.to_crs(epsg=4326)
            # Kürzel aus Spalte ermitteln: icc (lowercase), KANTONSNUM, name, etc.
            kab_col = None
            for col in gdf_raw.columns:
                sample = str(gdf_raw[col].dropna().iloc[0]) if len(gdf_raw)>0 else ''
                if col.lower() in ['icc','kab','abbreviation','abb']:
                    kab_col = col; break
                if col.lower() == 'kantonsnum':
                    kab_col = col; break
            if kab_col is None:
                for col in gdf_raw.columns:
                    vals = gdf_raw[col].dropna().astype(str).str.strip()
                    if vals.str.len().between(2,3).mean() > 0.8:
                        kab_col = col; break
            if kab_col:
                s = gdf_raw[kab_col].astype(str).str.strip()
                # Numerische Kantone (KANTONSNUM 1-26)
                if s.str.isnumeric().all():
                    gdf_raw['KAB'] = s.astype(int).map(KANT_NUM_TO_ABK)
                else:
                    # icc → uppercase (z.B. 'zh' → 'ZH')
                    gdf_raw['KAB'] = s.str.upper().str[:2]
                gdf_kant = gdf_raw[gdf_raw['KAB'].notna()].copy()
                print(f"Kantone geladen: {len(gdf_kant)} | Datei: {os.path.basename(path)} | Spalte: {kab_col}")
                break
        except Exception as e:
            print(f"  {os.path.basename(path)}: {e}")

if gdf_kant is None:
    print("⚠️  Kantonsgrenzen nicht gefunden — Animationen laufen mit CH-Umriss-Fallback.")

# Fallback: vereinfachter CH-Umriss als Polygon (WGS84, handdigitalisiert)
CH_OUTLINE = [
    (5.96,47.81),(6.44,47.87),(7.19,47.97),(7.61,47.59),(8.23,47.69),
    (8.62,47.76),(9.00,47.69),(9.53,47.52),(10.49,47.40),(10.38,47.01),
    (9.97,46.65),(9.56,46.49),(9.08,46.12),(8.86,46.07),(8.51,46.00),
    (8.08,46.11),(7.58,46.02),(7.06,45.92),(6.74,46.10),(6.34,46.44),
    (5.97,46.80),(5.96,47.17),(5.96,47.81)
]

print(f"Fallback CH-Umriss: {len(CH_OUTLINE)} Punkte")


In [ ]:
# ── Zonen-Zentroide & Geometrien ─────────────────────────────────────────────
# Handkalibrierte Zentroide der 6 K_01b-Zonen (WGS84)
# ⚙ Könnten aus gdf_kant automatisch berechnet werden (nach Config-Integration)
ZONE_CENTERS = {
    'Nordost'        : (8.68, 47.38),   # Zürich
    'Ostschweiz'     : (9.40, 46.78),   # Chur
    'Mitte-Erzeugung': (8.08, 47.40),   # Aarau
    'Mitte-Transit'  : (7.52, 46.95),   # Bern
    'West'           : (6.65, 46.55),   # Lausanne
    'Süd'            : (7.95, 46.25),   # Visp/Gotthard
}

# Cross-Border Grenzpunkte
BORDER_POINTS = {
    'DE': (7.60, 47.72),   # Basel
    'AT': (9.55, 47.48),   # Rheintal
    'IT': (8.95, 45.92),   # Chiasso / Domodossola
    'FR': (6.10, 46.22),   # Genf / Jura
}
BORDER_COLORS = {'DE':'#42A5F5','AT':'#EF5350','IT':'#66BB6A','FR':'#FFA726'}

# Interne Zonenfluss-Pfade (von_Zone, nach_Zone, Beschriftung)
# Richtung = normal; negatives MW = umgekehrt
ZONE_PAIRS = [
    ('Süd',             'Nordost',         'Göschenen-Airolo'),
    ('Süd',             'Mitte-Transit',   'Gotthard West'),
    ('Mitte-Erzeugung', 'Nordost',         'AKW-Export'),
    ('Mitte-Erzeugung', 'Mitte-Transit',   'Interne Verteilung'),
    ('Ostschweiz',      'Nordost',         'Ostschweiz→ZH'),
    ('West',            'Mitte-Transit',   'VD↔BE'),
]

# Zone-Polygon-Union (für farbige Hintergrundfelder)
gdf_zones_geo = None
KANTON_TO_ZONE_B = {
    'ZH':'Nordost','TG':'Nordost','SH':'Nordost','SG':'Nordost',
    'AR':'Ostschweiz','AI':'Ostschweiz','GR':'Ostschweiz','GL':'Ostschweiz','SZ':'Ostschweiz',
    'AG':'Mitte-Erzeugung','SO':'Mitte-Erzeugung','BL':'Mitte-Erzeugung',
    'BE':'Mitte-Transit','LU':'Mitte-Transit','BS':'Mitte-Transit',
    'ZG':'Mitte-Transit','NW':'Mitte-Transit','OW':'Mitte-Transit',
    'VD':'West','GE':'West','NE':'West','JU':'West','FR':'West',
    'VS':'Süd','TI':'Süd','UR':'Süd',
}

if gdf_kant is not None:
    try:
        # KAB = 2-letter canton code; try multiple column names
        kab_col = 'KAB' if 'KAB' in gdf_kant.columns else next((c for c in gdf_kant.columns if c.upper() in ['KANTONSNUM','ABBREVIATION','ABB']), None)
        gdf_kant['Zone_B'] = gdf_kant[kab_col].map(KANTON_TO_ZONE_B)
        gdf_zones_geo = (gdf_kant.dropna(subset=['Zone_B'])
                         .dissolve(by='Zone_B', as_index=False)
                         [['Zone_B','geometry']])
        print(f"Zonen-Polygone: {len(gdf_zones_geo)} Zonen aufgebaut")
    except Exception as e:
        print(f"Zonen-Union fehlgeschlagen: {e} — nur Zentroide werden genutzt")

print("Zentroide:")
for z, pos in ZONE_CENTERS.items():
    print(f"  {z:<16}: ({pos[0]:.2f}°E, {pos[1]:.2f}°N)")


---
## 2. Datenprofil-Modell<a id='profile_K_01c'></a>

[↑ Inhaltsverzeichnis](#toc_K_01c)

Synthetische stündliche und wöchentliche Profile. Alle Ausgaben in MW.
⚙ = Parameter die nach Config-Integration aus `config.json` kämen.

In [ ]:
# ── Zonenbasisdaten (aus K_01b-Logik) ────────────────────────────────────────
KANTON_POP = {
    'ZH':1593583,'BE':1065607,'LU':433654,'UR':37208,'SZ':165539,'OW':39028,
    'NW':43921,'GL':40590,'ZG':130426,'FR':340765,'SO':279375,'BS':183709,
    'BL':297025,'SH':86928,'AR':58050,'AI':16293,'SG':530468,'GR':202461,
    'AG':718084,'TG':295373,'TI':358353,'VD':826400,'VS':357043,'NE':177589,
    'GE':511655,'JU':73584
}
SPEZ_KW = 0.76  # ⚙ kW Mittellast/Person

# Mittlere Verbrauch + geschätzte Produktion pro Zone [MW]
ZONE_BASE = {}
for z in ZONE_COLORS_B:
    pop = sum(KANTON_POP.get(k,0) for k,v in KANTON_TO_ZONE_B.items() if v==z)
    ZONE_BASE[z] = {'pop': pop, 'verbrauch_mw': pop * SPEZ_KW / 1000}

# Geschätzte installierte Kapazitäten [MW installiert] — aus BFE-Kenntnis
# ⚙ Würde nach Config-Integration aus gdf_plants berechnet
ZONE_PROD_INSTALLED = {
    'Nordost':         {'Solar':1800,'Wasserkraft': 500,'Kernkraft':   0,'Andere': 200},
    'Ostschweiz':      {'Solar': 600,'Wasserkraft':2800,'Kernkraft':   0,'Andere': 100},
    'Mitte-Erzeugung': {'Solar':1200,'Wasserkraft': 800,'Kernkraft':5000,'Andere': 300},
    'Mitte-Transit':   {'Solar':1400,'Wasserkraft':1200,'Kernkraft':   0,'Andere': 400},
    'West':            {'Solar':1000,'Wasserkraft': 900,'Kernkraft':   0,'Andere': 150},
    'Süd':             {'Solar': 800,'Wasserkraft':8500,'Kernkraft':   0,'Andere':  50},
}
CF = {'Solar':0.12,'Wasserkraft':0.38,'Kernkraft':0.80,'Andere':0.40}

SAISONS = ['Winter','Frühling','Sommer','Herbst']
# Saisonale Kapazitätsfaktoren-Skalierung je ET
CF_SEASONAL = {
    'Solar':      {'Winter':0.30,'Frühling':0.75,'Sommer':1.00,'Herbst':0.50},
    'Wasserkraft':{'Winter':0.65,'Frühling':0.95,'Sommer':0.90,'Herbst':0.72},
    'Kernkraft':  {'Winter':1.00,'Frühling':0.98,'Sommer':0.92,'Herbst':0.98},
    'Andere':     {'Winter':1.10,'Frühling':1.00,'Sommer':0.85,'Herbst':1.00},
}
VERBRAUCH_SAISONAL = {'Winter':1.15,'Frühling':1.00,'Sommer':0.87,'Herbst':1.02}

HOURS = np.arange(24)
WEEKS = np.arange(52)

def solar_h(h):
    '''Solar-Tagesprofil normiert [0-1]'''
    return np.maximum(0.0, np.sin(np.pi * (h - 6.0) / 13.0))

def hydro_h(h):
    '''Wasserkraft-Tagesprofil (Dispatch-Charakteristik) normiert [0.7-1.3]'''
    return (1.0 + 0.22*np.exp(-((h-10.5)**2)/14)
                + 0.18*np.exp(-((h-19.0)**2)/10)
                - 0.10*np.exp(-((h-4.0)**2)/4))

def nuclear_h(h):
    return np.ones_like(np.array(h, dtype=float))

def consumption_h(h, saison='Frühling'):
    s = VERBRAUCH_SAISONAL.get(saison, 1.0)
    return s * (1.0 + 0.38*np.exp(-((h-8.5)**2)/4.5)
                    + 0.44*np.exp(-((h-19.0)**2)/5.0)
                    - 0.28*np.exp(-((h-4.0)**2)/3.0))

def zone_prod_mw(zone, h, saison):
    '''MW Produktion einer Zone bei Stunde h, Jahreszeit saison'''
    total = 0.0
    for et, inst_mw in ZONE_PROD_INSTALLED.get(zone, {}).items():
        cf_base = CF.get(et, 0.4)
        cf_s    = CF_SEASONAL.get(et, {}).get(saison, 1.0)
        if et == 'Solar':
            profile = solar_h(h)
        elif et == 'Wasserkraft':
            profile = hydro_h(h)
        elif et == 'Kernkraft':
            profile = nuclear_h(h)
        else:
            profile = 0.85  # Andere: flat
        total += inst_mw * cf_base * cf_s * profile
    return total

def zone_cons_mw(zone, h, saison):
    return ZONE_BASE[zone]['verbrauch_mw'] * consumption_h(h, saison)

def zone_imbalance_mw(zone, h, saison):
    return zone_prod_mw(zone, h, saison) - zone_cons_mw(zone, h, saison)

# Jahresprofil (52 Wochen) — saisonaler Verlauf
def solar_week(w):    return 0.12 + 0.10*np.sin(2*np.pi*(w-12)/52)
def hydro_week(w):    return 0.38 + 0.14*np.sin(2*np.pi*(w-15)/52)
def nuclear_week(w):  return 0.79 - 0.05*np.sin(2*np.pi*(w-26)/52)  # Sommerrevision
def cons_week(w):     return 1.00 - 0.14*np.sin(2*np.pi*(w-12)/52)  # Sommer-Tief

def zone_prod_mw_week(zone, w):
    total = 0.0
    for et, inst_mw in ZONE_PROD_INSTALLED.get(zone, {}).items():
        if et == 'Solar':        cf = solar_week(w)
        elif et == 'Wasserkraft': cf = hydro_week(w)
        elif et == 'Kernkraft':  cf = nuclear_week(w)
        else:                    cf = 0.40
        total += inst_mw * cf
    return total

def zone_cons_mw_week(zone, w):
    return ZONE_BASE[zone]['verbrauch_mw'] * cons_week(w)

def zone_imbalance_mw_week(zone, w):
    return zone_prod_mw_week(zone, w) - zone_cons_mw_week(zone, w)

# Precompute für Performance
print("Precomputing Stundendaten...")
HOURLY = {}
for saison in SAISONS:
    HOURLY[saison] = {}
    for zone in ZONE_COLORS_B:
        HOURLY[saison][zone] = {
            'prod': [zone_prod_mw(zone, h, saison) for h in HOURS],
            'cons': [zone_cons_mw(zone, h, saison) for h in HOURS],
            'imb':  [zone_imbalance_mw(zone, h, saison) for h in HOURS],
        }

print("Precomputing Jahresverlauf...")
WEEKLY = {}
for zone in ZONE_COLORS_B:
    WEEKLY[zone] = {
        'prod': [zone_prod_mw_week(zone, w) for w in WEEKS],
        'cons': [zone_cons_mw_week(zone, w) for w in WEEKS],
        'imb':  [zone_imbalance_mw_week(zone, w) for w in WEEKS],
    }

print("✅ Datenprofile bereit")
print(f"   Zonen: {list(ZONE_COLORS_B.keys())}")
print(f"   Jahreszeiten: {SAISONS}")
print(f"   Stunden: {len(HOURS)} | Wochen: {len(WEEKS)}")


---
## 3. Helper-Funktionen<a id='helper_K_01c'></a>

[↑ TOC](#toc_K_01c)

**🔎 Quellcode der importierten lib-Funktion**

Die Funktion `should_skip` wird aus `lib/plotting.py` importiert und ab
dieser Stelle im Notebook verwendet (intern in `make_gif_*`). Steuerung
über `sync/config.json` → `animation.modus` und `animation.overrides`.
Aufklappbar darunter ist der Quellcode einsehbar — ohne die `lib/`-Datei
öffnen zu müssen.


In [ ]:
show_source(should_skip)


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Helper: Interpolation + BFE-Cluster-Engine + Phase-Velocity-Dots + GIF-Builder
# ══════════════════════════════════════════════════════════════════════════════
import numpy as np
import io as _io
from scipy.interpolate import CubicSpline
from scipy.spatial import cKDTree
from PIL import Image as PILImage

MAP_XLIM = (5.88, 10.60)
MAP_YLIM = (45.78, 47.92)

TAG_TIMES  = np.linspace(0, 24, N_FRAMES_TAG,  endpoint=False)
JAHR_TIMES = np.linspace(0, 52, N_FRAMES_JAHR, endpoint=False)
MONAT_KURZ = ['Jan','Feb','Mrz','Apr','Mai','Jun','Jul','Aug','Sep','Okt','Nov','Dez']

# ── Splines ───────────────────────────────────────────────────────────────────
def make_spline_h(vals):
    h = np.arange(25); v = np.array(list(vals)+[vals[0]])
    return CubicSpline(h, v, bc_type='periodic')
def make_spline_w(vals):
    w = np.arange(53); v = np.array(list(vals)+[vals[0]])
    return CubicSpline(w, v, bc_type='periodic')

print("Berechne Splines...")
SPLINES_H = {}; SPLINES_W = {}
for saison in SAISONS:
    SPLINES_H[saison] = {}
    for zone in ZONE_COLORS_B:
        d = HOURLY[saison][zone]
        SPLINES_H[saison][zone] = {k: make_spline_h(d[k]) for k in ('prod','cons','imb')}
for zone in ZONE_COLORS_B:
    d = WEEKLY[zone]
    SPLINES_W[zone] = {k: make_spline_w(d[k]) for k in ('prod','cons','imb')}

# ── GIF A: BFE-Anlagen laden + clustern ──────────────────────────────────────
def _load_bfe_plants(data_dir):
    """
    Laedt BFE GeoPackage. SubCategory-Codes identisch mit K_01-Mapping.
    subcat_1=Wasserkraft, subcat_2=Solar, subcat_6=Kernkraft, etc.
    Gibt Liste von {et, lon, lat, mw} zurueck.
    """
    bfe_path = os.path.join(data_dir, 'bfe_produktionsanlagen.gpkg')
    if not os.path.exists(bfe_path): return None
    try:
        import geopandas as _gpd
        import pandas as _pd
        gdf = _gpd.read_file(bfe_path)
        if gdf.crs and gdf.crs.to_epsg() != 4326:
            gdf = gdf.to_crs(epsg=4326)
        gdf['lon'] = gdf.geometry.centroid.x
        gdf['lat'] = gdf.geometry.centroid.y

        # Identisch mit K_01 SUBCAT_MAP / MAINCAT_MAP
        SUBCAT_MAP = {
            'subcat_1':'Wasserkraft', 'subcat_2':'Solar',
            'subcat_3':'Andere',      'subcat_4':'Andere',
            'subcat_5':'Andere',      'subcat_6':'Kernkraft',
            'subcat_7':'Andere',      'subcat_8':'Andere',
            'subcat_9':'Andere',      'subcat_10':'Andere',
        }
        MAINCAT_MAP = {
            'maincat_1':'Wasserkraft', 'maincat_2':'Solar',
            'maincat_3':'Kernkraft',   'maincat_4':'Andere',
        }
        ET_TO_DISPLAY = {
            'Solar':'Solar', 'Wasserkraft':'Wasserkraft',
            'Kernkraft':'Kernkraft',
        }  # Rest -> Andere

        # Spalten finden (identisch mit K_01 find_col)
        def _find_col(*kws):
            for kw in kws:
                for col in gdf.columns:
                    if kw.lower() in col.lower(): return col
            return None

        subcat_col  = _find_col('subcat', 'subcategor')
        maincat_col = _find_col('maincat', 'maincategor')
        mw_col      = _find_col('totalpower', 'power', 'leistung')

        if mw_col is None:
            print(f"  Leistungsspalte nicht gefunden. Verfuegbar: {list(gdf.columns)}")
            return None

        # ET vektorisiert mappen
        _et = _pd.Series('Andere', index=gdf.index)
        if subcat_col:
            _mapped = (gdf[subcat_col].astype(str).str.strip().str.lower()
                       .map(SUBCAT_MAP))
            _et = _mapped.fillna(_et)
        if maincat_col:
            _mask = _et == 'Andere'
            if _mask.any():
                _mapped2 = (gdf.loc[_mask, maincat_col].astype(str)
                            .str.strip().str.lower().map(MAINCAT_MAP))
                _et[_mask] = _mapped2.fillna('Andere')

        gdf['ET_group'] = _et.map(lambda x: x if x in ET_TO_DISPLAY else 'Andere')
        gdf['kw']       = _pd.to_numeric(gdf[mw_col], errors='coerce').fillna(0)

        # Filter: nur CH-Bounding-Box (Polygon-Filter folgt nach gdf_kant-Load)
        mask_ch = ((gdf['lon'] > 5.88) & (gdf['lon'] < 10.65) &
                   (gdf['lat'] > 45.78) & (gdf['lat'] < 47.95) &
                   (gdf['kw'] > 0.5))
        gdf = gdf[mask_ch]

        plants = [
            {'et': row['ET_group'], 'lon': float(row['lon']),
             'lat': float(row['lat']), 'mw': float(row['kw'])/1000}
            for _, row in gdf.iterrows()
            if row['kw'] > 0.5
        ]
        et_counts = {}
        for p in plants: et_counts[p['et']] = et_counts.get(p['et'],0)+1
        print(f"  ET-Verteilung: {et_counts}")
        return plants if len(plants) > 10 else None
    except Exception as e:
        print(f"  BFE-Ladefehler: {e}"); return None


def cluster_plants(plants_list, cluster_deg):
    if not plants_list: return []
    coords = np.array([[p['lon'],p['lat']] for p in plants_list])
    mws    = np.array([p['mw'] for p in plants_list])
    tree   = cKDTree(coords)
    visited= np.zeros(len(plants_list), bool)
    result = []
    for i in range(len(plants_list)):
        if visited[i]: continue
        members = np.array(tree.query_ball_point(coords[i], cluster_deg))
        visited[members] = True
        total = mws[members].sum()
        clon  = np.average(coords[members,0], weights=mws[members])
        clat  = np.average(coords[members,1], weights=mws[members])
        result.append({'et':plants_list[i]['et'],'lon':float(clon),
                       'lat':float(clat),'mw':float(total)})
    return result

print("Lade BFE-Anlagen...", end=' ')
_raw_plants = _load_bfe_plants(PROD_DATA_DIR)
if _raw_plants is None:
    _bfe_path = os.path.join(PROD_DATA_DIR, 'bfe_produktionsanlagen.gpkg')
    raise FileNotFoundError(
        f"BFE-Anlagen nicht gefunden: {_bfe_path}\n"
        "Bitte zuerst K_01_Raeumliche_Analyse.ipynb ausfuehren.")
print(f"{len(_raw_plants)} Anlagen aus BFE geladen")

_plants_by_et = {et:[p for p in _raw_plants if p['et']==et] for et in ET_COLORS}
CLUSTERS = {et: cluster_plants(_plants_by_et[et], GIF_A_CLUSTER_DEG) for et in ET_COLORS}
ET_MAX_MW= {et: max((c['mw'] for c in CLUSTERS[et]),default=1) for et in ET_COLORS}
print(f"Cluster (r={GIF_A_CLUSTER_DEG}°): " +
      " | ".join(f"{et}:{len(CLUSTERS[et])}" for et in ET_COLORS))

# ── Kapazitätsfaktor ──────────────────────────────────────────────────────────
def get_cf(et, t_h, saison='Sommer'):
    base = CF_SEASONAL.get(et,{}).get(saison, 1.0)
    if et=='Solar':
        return base * max(0.0, np.sin(np.pi*(t_h-6.0)/13.0))
    if et=='Wasserkraft':
        return CF.get(et,0.4) * base * (0.88 + 0.12*np.sin(np.pi*(t_h-6)/24))
    if et=='Kernkraft': return CF.get(et,0.8) * base
    return CF.get(et,0.4) * base * 0.90

def get_cf_w(et, t_w):
    """
    Kapazitaetsfaktor fuer Jahresverlauf (Woche t_w, 0..52).
    Gibt den Tages-Durchschnitt-CF zurueck — sinusoidal interpoliert
    zwischen Winter-Minimum und Sommer-Maximum.
    Solar: Winter fast 0, Sommer deutlich heller.
    Kernkraft: nahezu konstant.
    """
    import numpy as _np_cf
    # Saisonale Phase: 0=Winter(Woche 0), 1=Sommer(Woche 26)
    # sin(2pi*(t_w-13)/52): -1 bei Woche 0, +1 bei Woche 26
    phase = float(_np_cf.sin(2 * _np_cf.pi * (t_w - 13) / 52))
    alpha_blend = (phase + 1) / 2  # 0=Winter, 1=Sommer

    # Tages-Ø CF pro Saison: integral(get_cf over 24h) / 24
    _hrs = _np_cf.linspace(0, 24, 48, endpoint=False)
    def _day_avg(saison):
        return float(_np_cf.mean([get_cf(et, h, saison) for h in _hrs]))

    cf_winter = _day_avg('Winter')
    cf_sommer = _day_avg('Sommer')

    # Linear interpoliert zwischen Winter-Ø und Sommer-Ø
    cf = cf_winter + (cf_sommer - cf_winter) * alpha_blend
    return float(_np_cf.clip(cf, 0, 1))


# ── Erzeuger-Dots ─────────────────────────────────────────────────────────────
def draw_generators(ax, cf_fn):
    for et, col in ET_COLORS.items():
        cf    = cf_fn(et)
        alpha = GIF_A_ALPHA_MIN + cf*(GIF_A_ALPHA_MAX - GIF_A_ALPHA_MIN)
        if alpha < 0.03: continue
        for c in CLUSTERS.get(et,[]):
            norm = np.sqrt(c['mw'] / ET_MAX_MW[et])
            s    = max(6, norm * GIF_A_SIZE_SCALE)
            ax.scatter(c['lon'], c['lat'], s=s, c=col,
                       alpha=alpha, zorder=12, linewidths=0, rasterized=True)

# ── Phase-Velocity Flow-Dots ──────────────────────────────────────────────────
def _get_mw_norm(zone_from, zone_to, mw_flow, path_norms):
    if GIF_C_NORM_MODE == 'global': return GIF_C_MW_NORM_GLOBAL
    return max(path_norms.get((zone_from,zone_to), abs(mw_flow)), 100)

def draw_flow_dots(ax, zone_from, zone_to, mw_flow, frame_idx, n_frames,
                   mw_norm, min_mw=60):
    if abs(mw_flow) < min_mw: return
    p1 = np.array(ZONE_CENTERS[zone_from])
    p2 = np.array(ZONE_CENTERS[zone_to])
    phase_vel = mw_flow / mw_norm
    rel_flow  = abs(mw_flow) / mw_norm
    n_dots    = max(1, min(8, int(rel_flow * GIF_C_N_DOTS_BASE) + 1))
    col = ZONE_COLORS_B[zone_from] if mw_flow >= 0 else ZONE_COLORS_B[zone_to]
    phase = frame_idx * phase_vel / n_frames
    for i in range(n_dots):
        t   = (phase + i/n_dots) % 1.0
        pos = p1 + t*(p2-p1)
        ax.scatter(*pos, s=GIF_C_DOT_SIZE, c=col,
                   alpha=0.88, zorder=10, linewidths=0, rasterized=True)
    ax.plot([p1[0],p2[0]],[p1[1],p2[1]], color=col, lw=0.5, alpha=0.18, zorder=5)
    if GIF_C_SHOW_MW_LABEL:
        mid = (p1+p2)/2
        ax.text(mid[0], mid[1]+0.04, f'{abs(mw_flow):.0f}',
                ha='center', va='bottom', color=col, fontsize=5, fontweight='bold',
                bbox=dict(boxstyle='round,pad=0.06',fc='#090d14',alpha=0.6,lw=0), zorder=14)

def draw_border_dots(ax, neighbor, mw_flow, frame_idx, n_frames,
                     mw_norm, ch_center=(8.15,46.90), min_mw=50):
    if abs(mw_flow) < min_mw: return
    bp  = BORDER_POINTS[neighbor]
    col = BORDER_COLORS[neighbor]
    p1  = np.array(ch_center); p2 = np.array(bp)
    if mw_flow < 0: p1,p2 = p2,p1
    rel   = abs(mw_flow)/max(mw_norm,100)
    n_dots= max(1, min(6, int(rel*GIF_C_N_DOTS_BASE)))
    phase = frame_idx*(mw_flow/mw_norm)/n_frames
    for i in range(n_dots):
        t   = (phase + i/n_dots) % 1.0
        pos = p1 + t*(p2-p1)
        ax.scatter(*pos, s=GIF_C_DOT_SIZE*0.7, c=col,
                   alpha=0.88, zorder=11, linewidths=0, rasterized=True)
    ax.plot([ch_center[0],bp[0]],[ch_center[1],bp[1]],
            color=col, lw=0.5, alpha=0.18, zorder=4)
    ax.text(bp[0],bp[1]+0.12, f"{neighbor}\n{mw_flow:+.0f}",
            ha='center',va='bottom',color=col,fontsize=6,fontweight='bold',zorder=15,
            bbox=dict(boxstyle='round,pad=0.12',fc='#090d14',alpha=0.75,lw=0))

# ── Basiskarte ────────────────────────────────────────────────────────────────
def draw_base_map(ax, zone_alpha=0.12):
    ax.set_xlim(*MAP_XLIM); ax.set_ylim(*MAP_YLIM)
    ax.set_facecolor('#090d14'); ax.set_axis_off()
    if gdf_zones_geo is not None:
        import geopandas as _gpd
        for _, row in gdf_zones_geo.iterrows():
            _gpd.GeoDataFrame([row],geometry='geometry',crs='EPSG:4326').plot(
                ax=ax,color=ZONE_COLORS_B.get(row['Zone_B'],'#555'),
                alpha=zone_alpha,linewidth=0,zorder=1)
    if gdf_kant is not None:
        gdf_kant.boundary.plot(ax=ax,color='#2a3040',linewidth=0.6,zorder=2)
        try:
            import geopandas as _gpd
            _gpd.GeoDataFrame(geometry=[gdf_kant.unary_union.boundary],
                              crs='EPSG:4326').plot(ax=ax,color='#445566',lw=1.8,zorder=3)
        except Exception: pass
    for z,(lon,lat) in ZONE_CENTERS.items():
        short=z.replace('Mitte-','M-').replace('Ostschweiz','Ostschw.')
        ax.text(lon,lat+0.08,short,ha='center',va='bottom',
                color=ZONE_COLORS_B[z],fontsize=7,fontweight='bold',zorder=15,
                bbox=dict(boxstyle='round,pad=0.15',fc='#090d14',alpha=0.7,lw=0))

def add_timestamp_bar(ax, left, center, right):
    xs=MAP_XLIM[1]-MAP_XLIM[0]; ys=MAP_YLIM[1]-MAP_YLIM[0]
    ax.text(MAP_XLIM[0]+0.01*xs,MAP_YLIM[1]-0.02*ys,left,
            ha='left',va='top',color='#aaa',fontsize=8,zorder=20)
    ax.text(MAP_XLIM[0]+0.5*xs,MAP_YLIM[1]-0.02*ys,center,
            ha='center',va='top',color='white',fontsize=9,fontweight='bold',zorder=20)
    ax.text(MAP_XLIM[1]-0.01*xs,MAP_YLIM[1]-0.02*ys,right,
            ha='right',va='top',color='#ff5252',fontsize=7,zorder=20)

# ── Hintergrund-Cache + GIF-Builder ──────────────────────────────────────────
_BG_CACHE = {}

def _render_background(zone_alpha=0.12):
    if zone_alpha in _BG_CACHE: return _BG_CACHE[zone_alpha]
    print(f"  Pre-render Hintergrund...", end=' ')
    import matplotlib.pyplot as _plt
    fig = _plt.figure(figsize=(14,9),facecolor='#0d1117')
    ax  = fig.add_axes([0,0,1,1])
    draw_base_map(ax, zone_alpha)
    buf = _io.BytesIO()
    fig.savefig(buf,format='png',dpi=DPI_GIF,facecolor='#0d1117')
    _plt.close(fig); buf.seek(0)
    img = PILImage.open(buf).convert('RGBA').copy()
    _BG_CACHE[zone_alpha] = img
    print(f"OK ({img.size[0]}x{img.size[1]}px)")
    return img

def make_gif_fast(draw_fn, n_frames, fps, path, zone_alpha=0.12):
    # Skip-Check: überspringt Erzeugung wenn GIF existiert
    # (gesteuert durch sync/config.json → animation.modus)
    _name = os.path.basename(path).rsplit('.', 1)[0]
    if should_skip(path, 'animation', _name, CFG):
        print(f'⏭️  {_name} übersprungen (existiert)')
        return
    bg = _render_background(zone_alpha)
    frame_dir = path.replace('.gif','_frames')
    os.makedirs(frame_dir, exist_ok=True)
    frames_pil = []
    import matplotlib.pyplot as _plt
    for fi in range(n_frames):
        fig = _plt.figure(figsize=(14,9),facecolor=(0,0,0,0))
        ax  = fig.add_axes([0,0,1,1])
        ax.set_xlim(*MAP_XLIM); ax.set_ylim(*MAP_YLIM)
        ax.set_facecolor((0,0,0,0)); ax.patch.set_alpha(0); ax.set_axis_off()
        draw_fn(ax, fi, n_frames)
        buf = _io.BytesIO()
        fig.savefig(buf,format='png',dpi=DPI_GIF,facecolor=(0,0,0,0),
                    transparent=True,bbox_inches=None,pad_inches=0)
        _plt.close(fig); buf.seek(0)
        ov = PILImage.open(buf).convert('RGBA').copy()
        if bg.size!=ov.size: ov=ov.resize(bg.size,PILImage.LANCZOS)
        fr = bg.copy(); fr.paste(ov,mask=ov.split()[3])
        fr = fr.convert('RGB')
        fr.save(os.path.join(frame_dir,f'frame_{fi:04d}.png'),optimize=True)
        frames_pil.append(fr)
    frames_pil[0].save(path,save_all=True,append_images=frames_pil[1:],
                       duration=int(1000/fps),loop=0,optimize=True)
    kb=os.path.getsize(path)//1024
    print(f"  {os.path.basename(path)} ({n_frames}f @{fps}fps={n_frames/fps:.1f}s | {kb} KB)")
    print(f"   Frames: {frame_dir}/")

# ── Pfad-Normierung vorberechnen (per_path Modus) ────────────────────────────
_PATH_NORMS = {}
for _zf,_zt,_ in ZONE_PAIRS:
    _vals = [abs(float(SPLINES_H[s][_zf]['imb'](h)) - float(SPLINES_H[s][_zt]['imb'](h)))/2
             for s in SAISONS for h in np.linspace(0,24,48)]
    _PATH_NORMS[(_zf,_zt)] = max(float(np.mean(_vals)), 150)
PATH_NORMS = _PATH_NORMS
BORDER_NORM= max(float(np.mean([abs(float(SPLINES_H[s][z]['imb'](12)))
                                 for s in SAISONS for z in ZONE_COLORS_B]))*0.3, 200)

# NEIGHBOR_WEIGHT + CH_CENTER (brauchen GIF D/E)
CH_CENTER       = (8.15, 46.90)
NEIGHBOR_WEIGHT = {'DE':0.35,'AT':0.15,'IT':0.30,'FR':0.20}

print(f"Helpers geladen | norm='{GIF_C_NORM_MODE}' | "
      f"Cluster r={GIF_A_CLUSTER_DEG} | {N_FRAMES_TAG}f Tag, {N_FRAMES_JAHR}f Jahr")


---
## 4. GIF A — Erzeugungsschwankungen<a id='gif_a_K_01c'></a>

[↑ Inhaltsverzeichnis](#toc_K_01c)

Dots pro Zone & Energieträger. Grösse = aktuelle Produktions-MW. Tagesverlauf: 4× (je Jahreszeit, 24 Frames). Jahresverlauf: 52 Frames.

In [ ]:
print("\n▶ Starte Erstellung: GIF A Tag — BFE-Anlagen, Alpha=CF (1/10)")
# 📡 BFE-Anlagenpositionen | Alpha = aktueller Kapazitätsfaktor | Grösse = Kapazität
# ⚙ Schalter: GIF_A_CLUSTER_DEG, GIF_A_SIZE_SCALE, GIF_A_ALPHA_MIN, GIF_A_ALPHA_MAX
import matplotlib.patches as _mp_a

def _dyn_a_tag(ax, fi, nf, saison):
    t_h   = TAG_TIMES[fi]; h_disp = int(round(t_h))%24
    draw_generators(ax, cf_fn=lambda et: get_cf(et, t_h, saison))
    ax.legend(handles=[_mp_a.Patch(color=ET_COLORS[et], label=et)
                        for et in ['Solar','Wasserkraft','Kernkraft','Andere']],
              loc='lower left', fontsize=6, framealpha=0.5,
              facecolor='#111', labelcolor='white',
              bbox_to_anchor=(0.01,0.01), bbox_transform=ax.transAxes)
    add_timestamp_bar(ax, f"⚙ Modelliert | {saison}",
                      f"Erzeugung — {h_disp:02d}:00 Uhr",
                      "⚠️ Synthetische Daten")

for saison in SAISONS:
    path = os.path.join(EXP_CHARTS_DIR, f"EXP_kuer_k01c_gif_a_tag_{saison.lower()}.gif")
    make_gif_fast(lambda ax,fi,nf,s=saison: _dyn_a_tag(ax,fi,nf,s),
                  N_FRAMES_TAG, FPS_TAG, path, zone_alpha=0.08)


**🔎 Quellcode der importierten lib-Funktion**

Die Funktion `show_animation` wird aus `lib/widgets.py` importiert und ab
dieser Stelle im Notebook verwendet. Mit einem Klick auf den aufklappbaren
Block darunter ist der Quellcode direkt im Notebook einsehbar — ohne die
`lib/`-Datei öffnen zu müssen.


In [ ]:
show_source(show_animation)


In [ ]:
# ── Ausgabe anzeigen ────────────────────────────────────────────
_p = os.path.join(EXP_CHARTS_DIR, 'EXP_kuer_k01c_gif_a_tag_winter.gif')
# Umschalten: mode='gif' (Standbild) oder mode='slider' (interaktiv).
# framerate nur für Slider relevant (bei GIF steckt fps im GIF selbst).
show_animation(_p, mode='gif', framerate=10)


**GIF A — Erzeugung Tagesverlauf (Winter)**

Dots nach Energieträger: Solar=gelb, Wasserkraft=blau, Kernkraft=violett.
Grösse = aktuelle MW. Solar ist bei 00:00 Uhr Null — gut sichtbar ab Frame 6 (06:00).

*Alle 4 GIF(s) dieses Typs liegen in `output/charts/experimental/`.*

In [ ]:
print("\n▶ Starte Erstellung: GIF A Jahr — BFE-Anlagen, Alpha=CF saisonal (2/10)")
# 📡 BFE-Anlagenpositionen | Alpha = saisonaler Kapazitätsfaktor
import matplotlib.patches as _mp_aj

def _dyn_a_jahr(ax, fi, nf):
    t_w   = JAHR_TIMES[fi]; monat = MONAT_KURZ[int(t_w/52*12)%12]
    draw_generators(ax, cf_fn=lambda et: get_cf_w(et, t_w))
    ax.legend(handles=[_mp_aj.Patch(color=ET_COLORS[et], label=et)
                        for et in ['Solar','Wasserkraft','Kernkraft','Andere']],
              loc='lower left', fontsize=6, framealpha=0.5,
              facecolor='#111', labelcolor='white',
              bbox_to_anchor=(0.01,0.01), bbox_transform=ax.transAxes)
    add_timestamp_bar(ax, "⚙ Modelliert | Ø Tageswert",
                      f"Erzeugung — Woche {int(t_w)+1:02d}/52 ({monat})",
                      "⚠️ Synthetische Daten")

path = os.path.join(EXP_CHARTS_DIR, 'EXP_kuer_k01c_gif_a_jahr.gif')
make_gif_fast(_dyn_a_jahr, N_FRAMES_JAHR, FPS_JAHR, path, zone_alpha=0.08)


In [ ]:
# ── Ausgabe anzeigen ────────────────────────────────────────────
_p = os.path.join(EXP_CHARTS_DIR, 'EXP_kuer_k01c_gif_a_jahr.gif')
# Umschalten: mode='gif' (Standbild) oder mode='slider' (interaktiv).
# framerate nur für Slider relevant (bei GIF steckt fps im GIF selbst).
show_animation(_p, mode='gif', framerate=10)


**GIF A — Erzeugung Jahresverlauf**

52 Wochen. Solar: Sommer-Peak. Wasserkraft: Schneeschmelze Frühling. Kernkraft: flach.

*Alle 1 GIF(s) dieses Typs liegen in `output/charts/experimental/`.*

---
## 5. GIF B — Verbrauchsschwankungen<a id='gif_b_K_01c'></a>

[↑ Inhaltsverzeichnis](#toc_K_01c)

Farbige Dots (Zonenfarbe) an Zonenzentroiden. Grösse = aktuelle Last-MW.

In [ ]:
print("\n▶ Starte Erstellung: GIF B Tag (3/10)")
def _dynamic_b_tag(ax, fi, nf, saison):
    t_h = TAG_TIMES[fi]; h_disp = int(round(t_h)) % 24
    for zone, (cx, cy) in ZONE_CENTERS.items():
        mw_cons = float(SPLINES_H[saison][zone]['cons'](t_h))
        imb     = float(SPLINES_H[saison][zone]['imb'](t_h))
        ax.scatter(cx, cy, s=min(700, max(40, mw_cons/4)),
                   c=ZONE_COLORS_B[zone], alpha=0.70, zorder=10, linewidths=0, rasterized=True)
        ring_col = '#66BB6A' if imb > 0 else '#EF5350'
        ax.scatter(cx, cy, s=min(800, max(80, abs(imb)/3)),
                   facecolors='none', edgecolors=ring_col, linewidths=2.5, alpha=0.65, zorder=11)
        ax.text(cx, cy-0.14, f'{mw_cons:.0f} MW', ha='center', va='top',
                color='white', fontsize=5.5, zorder=13,
                bbox=dict(boxstyle='round,pad=0.1', fc='#111', alpha=0.6, lw=0))
    import matplotlib.patches as _mp
    ax.legend(handles=[_mp.Patch(color='#66BB6A', label='Überschuss'),
               _mp.Patch(color='#EF5350', label='Defizit')],
              loc='lower left', fontsize=6, framealpha=0.5,
              facecolor='#111', labelcolor='white',
              bbox_to_anchor=(0.01,0.01), bbox_transform=ax.transAxes)
    add_timestamp_bar(ax, f"⚙ Modelliert | {saison}",
                      f"Verbrauch + Imbalance — {h_disp:02d}:00 Uhr",
                      "⚠️ Synthetische Daten")

for saison in SAISONS:
    path = os.path.join(EXP_CHARTS_DIR, f"EXP_kuer_k01c_gif_b_tag_{saison.lower()}.gif")
    make_gif_fast(lambda ax,fi,nf,s=saison: _dynamic_b_tag(ax,fi,nf,s),
                  N_FRAMES_TAG, FPS_TAG, path, zone_alpha=0.06)


In [ ]:
# ── Ausgabe anzeigen ────────────────────────────────────────────
_p = os.path.join(EXP_CHARTS_DIR, 'EXP_kuer_k01c_gif_b_tag_winter.gif')
# Umschalten: mode='gif' (Standbild) oder mode='slider' (interaktiv).
# framerate nur für Slider relevant (bei GIF steckt fps im GIF selbst).
show_animation(_p, mode='gif', framerate=10)


**GIF B — Verbrauch Tagesverlauf (Winter)**

Dot-Grösse = Mittellast [MW]. Ring grün = Überschuss, rot = Defizit.
Nordost (ZH) zeigt im Winter dauerhaftes Defizit (grösster Verbraucher CH).

*Alle 4 GIF(s) dieses Typs liegen in `output/charts/experimental/`.*

In [ ]:
print("\n▶ Starte Erstellung: GIF B Jahr (4/10)")
def _dynamic_b_jahr(ax, fi, nf):
    t_w = JAHR_TIMES[fi]; monat = MONAT_KURZ[int(t_w/52*12)%12]
    for zone, (cx, cy) in ZONE_CENTERS.items():
        mw_cons = float(SPLINES_W[zone]['cons'](t_w))
        imb     = float(SPLINES_W[zone]['imb'](t_w))
        ax.scatter(cx, cy, s=min(800, max(50, mw_cons/4)),
                   c=ZONE_COLORS_B[zone], alpha=0.70, zorder=10, linewidths=0, rasterized=True)
        ring_col = '#66BB6A' if imb > 0 else '#EF5350'
        ax.scatter(cx, cy, s=min(900, max(80, abs(imb)/3)),
                   facecolors='none', edgecolors=ring_col, linewidths=2.5, alpha=0.65, zorder=11)
        ax.text(cx, cy-0.14, f'{mw_cons:.0f} MW', ha='center', va='top',
                color='white', fontsize=5.5, zorder=13,
                bbox=dict(boxstyle='round,pad=0.1', fc='#111', alpha=0.6, lw=0))
    add_timestamp_bar(ax, "⚙ Modelliert | Ø Tageswert",
                      f"Verbrauch — Woche {int(t_w)+1:02d}/52 ({monat})",
                      "⚠️ Synthetische Daten")

path = os.path.join(EXP_CHARTS_DIR, 'EXP_kuer_k01c_gif_b_jahr.gif')
make_gif_fast(_dynamic_b_jahr, N_FRAMES_JAHR, FPS_JAHR, path, zone_alpha=0.06)


In [ ]:
# ── Ausgabe anzeigen ────────────────────────────────────────────
_p = os.path.join(EXP_CHARTS_DIR, 'EXP_kuer_k01c_gif_b_jahr.gif')
# Umschalten: mode='gif' (Standbild) oder mode='slider' (interaktiv).
# framerate nur für Slider relevant (bei GIF steckt fps im GIF selbst).
show_animation(_p, mode='gif', framerate=10)


**GIF B — Verbrauch Jahresverlauf**

Winter-Peak Woche 1–10 und 44–52 deutlich. Sommer-Tief Woche 26.

*Alle 1 GIF(s) dieses Typs liegen in `output/charts/experimental/`.*

---
## 6. GIF C — Zonenfluss<a id='gif_c_K_01c'></a>

[↑ Inhaltsverzeichnis](#toc_K_01c)

Moving Dots auf Pfaden zwischen Zonen. Richtung = Flussrichtung. Dot-Dichte & Grösse = MW-Intensität.

In [ ]:
print("\n▶ Starte Erstellung: GIF C Tag — Phase-Velocity Dots (5/10)")
# 🔬 Zonenfluss | Phase-Velocity | Richtungswechsel fliessend
# ⚙ GIF_C_NORM_MODE: 'global' oder 'per_path'

def _dyn_c(ax, fi, nf, get_imb_fn, label):
    for zf,zt,_ in ZONE_PAIRS:
        imb_f=get_imb_fn(zf); imb_t=get_imb_fn(zt)
        if imb_f>0 and imb_t<0:   mw=min(abs(imb_f),abs(imb_t))
        elif imb_t>0 and imb_f<0: mw=-min(abs(imb_t),abs(imb_f))
        else:
            mw=imb_f-imb_t
            if abs(mw)<60: continue
        draw_flow_dots(ax,zf,zt,mw,fi,nf,_get_mw_norm(zf,zt,mw,PATH_NORMS))
    for zone,(cx,cy) in ZONE_CENTERS.items():
        imb=get_imb_fn(zone)
        col='#66BB6A' if imb>0 else '#EF5350'
        ax.scatter(cx,cy,s=max(40,min(300,abs(imb)/5)),c=col,alpha=0.50,zorder=7,linewidths=0)
        ax.text(cx,cy,f'{imb:+.0f}',ha='center',va='center',
                color='white',fontsize=5,fontweight='bold',zorder=14)
    add_timestamp_bar(ax,f"norm='{GIF_C_NORM_MODE}'",label,"⚠️ Synthetische Daten")

for saison in SAISONS:
    def make_c(s):
        def fn(ax,fi,nf):
            t_h=TAG_TIMES[fi]; h=int(round(t_h))%24
            _dyn_c(ax,fi,nf,lambda z:float(SPLINES_H[s][z]['imb'](t_h)),
                   f"Zonenfluss — {h:02d}:00 ({s})")
        return fn
    path = os.path.join(EXP_CHARTS_DIR, f"EXP_kuer_k01c_gif_c_tag_{saison.lower()}.gif")
    make_gif_fast(make_c(saison), N_FRAMES_TAG, FPS_TAG, path, zone_alpha=0.10)


In [ ]:
# ── Ausgabe anzeigen ────────────────────────────────────────────
_p = os.path.join(EXP_CHARTS_DIR, 'EXP_kuer_k01c_gif_c_tag_frühling.gif')
# Umschalten: mode='gif' (Standbild) oder mode='slider' (interaktiv).
# framerate nur für Slider relevant (bei GIF steckt fps im GIF selbst).
show_animation(_p, mode='gif', framerate=10)


**GIF C — Zonenfluss Tagesverlauf (Frühling)**

Moving Dots auf Pfaden. Hauptkorridor: Süd → Nordost (Göschenen-Achse).
Frühling zeigt stärksten Fluss (Schneeschmelze + tiefer Verbrauch).

*Alle 4 GIF(s) dieses Typs liegen in `output/charts/experimental/`.*

In [ ]:
print("\n▶ Starte Erstellung: GIF C Jahr — Phase-Velocity Dots (6/10)")
def _dyn_c_j(ax,fi,nf):
    t_w=JAHR_TIMES[fi]; m=MONAT_KURZ[int(t_w/52*12)%12]
    _dyn_c(ax,fi,nf,lambda z:float(SPLINES_W[z]['imb'](t_w)),
           f"Zonenfluss — Woche {int(t_w)+1:02d}/52 ({m})")
path=os.path.join(EXP_CHARTS_DIR,'EXP_kuer_k01c_gif_c_jahr.gif')
make_gif_fast(_dyn_c_j, N_FRAMES_JAHR, FPS_JAHR, path, zone_alpha=0.10)


In [ ]:
# ── Ausgabe anzeigen ────────────────────────────────────────────
_p = os.path.join(EXP_CHARTS_DIR, 'EXP_kuer_k01c_gif_c_jahr.gif')
# Umschalten: mode='gif' (Standbild) oder mode='slider' (interaktiv).
# framerate nur für Slider relevant (bei GIF steckt fps im GIF selbst).
show_animation(_p, mode='gif', framerate=10)


**GIF C — Zonenfluss Jahresverlauf**

Jahresverlauf der Zonenflüsse. Intensität folgt Wasserkraft-Profil.

*Alle 1 GIF(s) dieses Typs liegen in `output/charts/experimental/`.*

---
## 7. GIF D — Cross-Border Import/Export<a id='gif_d_K_01c'></a>

[↑ TOC](#toc_K_01c)

Moving Dots zwischen CH-Zentrum und Grenzpunkten DE/AT/IT/FR. Richtung = Export (raus) / Import (rein).

In [ ]:
print("\n▶ Starte Erstellung: GIF D Tag — Cross-Border Phase-Velocity (7/10)")
# 🔬 Cross-Border | ⚙ GIF_C_NORM_MODE gilt auch hier
import matplotlib.patches as _mp_d

def _dyn_d(ax, fi, nf, get_imb_fn, label):
    total = sum(get_imb_fn(z) for z in ZONE_COLORS_B)
    for nb_key,mw in {nb:total*w for nb,w in NEIGHBOR_WEIGHT.items()}.items():
        mw_n = BORDER_NORM if GIF_C_NORM_MODE=='global' else max(abs(mw),100)
        draw_border_dots(ax,nb_key,mw,fi,nf,mw_n,CH_CENTER)
    col_s='#66BB6A' if total>0 else '#EF5350'
    ax.scatter(*CH_CENTER,s=180,c=col_s,alpha=0.65,zorder=13,linewidths=0)
    ax.text(CH_CENTER[0],CH_CENTER[1]+0.18,f"CH\n{total:+.0f} MW",
            ha='center',va='bottom',color=col_s,fontsize=6.5,fontweight='bold',zorder=16,
            bbox=dict(boxstyle='round,pad=0.2',fc='#090d14',alpha=0.85,lw=0))
    ax.legend(handles=[_mp_d.Patch(color=BORDER_COLORS[n],label=f'CH-{n}')
                        for n in BORDER_POINTS],
              loc='lower left',fontsize=6,framealpha=0.5,facecolor='#111',labelcolor='white',
              bbox_to_anchor=(0.01,0.01),bbox_transform=ax.transAxes)
    add_timestamp_bar(ax,f"norm='{GIF_C_NORM_MODE}'",label,"⚠️ Synthetische Daten")

for saison in SAISONS:
    def make_d(s):
        def fn(ax,fi,nf):
            t_h=TAG_TIMES[fi]; h=int(round(t_h))%24
            _dyn_d(ax,fi,nf,lambda z:float(SPLINES_H[s][z]['imb'](t_h)),
                   f"Cross-Border — {h:02d}:00 ({s})")
        return fn
    path=os.path.join(EXP_CHARTS_DIR,f"EXP_kuer_k01c_gif_d_tag_{saison.lower()}.gif")
    make_gif_fast(make_d(saison), N_FRAMES_TAG, FPS_TAG, path, zone_alpha=0.06)


In [ ]:
# ── Ausgabe anzeigen ────────────────────────────────────────────
_p = os.path.join(EXP_CHARTS_DIR, 'EXP_kuer_k01c_gif_d_tag_sommer.gif')
# Umschalten: mode='gif' (Standbild) oder mode='slider' (interaktiv).
# framerate nur für Slider relevant (bei GIF steckt fps im GIF selbst).
show_animation(_p, mode='gif', framerate=10)


**GIF D — Cross-Border Tagesverlauf (Sommer)**

CH exportiert im Sommer hauptsächlich nach IT (30%) und DE (35%).
Dots wandern von CH-Zentrum zu Grenzpunkten (Laufenburg, Castione, Romanel, Pradella).

*Alle 4 GIF(s) dieses Typs liegen in `output/charts/experimental/`.*

In [ ]:
print("\n▶ Starte Erstellung: GIF D Jahr — Cross-Border Phase-Velocity (8/10)")
def _dyn_d_j(ax,fi,nf):
    t_w=JAHR_TIMES[fi]; m=MONAT_KURZ[int(t_w/52*12)%12]
    _dyn_d(ax,fi,nf,lambda z:float(SPLINES_W[z]['imb'](t_w)),
           f"Cross-Border — Woche {int(t_w)+1:02d}/52 ({m})")
path=os.path.join(EXP_CHARTS_DIR,'EXP_kuer_k01c_gif_d_jahr.gif')
make_gif_fast(_dyn_d_j, N_FRAMES_JAHR, FPS_JAHR, path, zone_alpha=0.06)


In [ ]:
# ── Ausgabe anzeigen ────────────────────────────────────────────
_p = os.path.join(EXP_CHARTS_DIR, 'EXP_kuer_k01c_gif_d_jahr.gif')
# Umschalten: mode='gif' (Standbild) oder mode='slider' (interaktiv).
# framerate nur für Slider relevant (bei GIF steckt fps im GIF selbst).
show_animation(_p, mode='gif', framerate=10)


**GIF D — Cross-Border Jahresverlauf**

CH-Nettosaldo übers Jahr. Sommer: Export. Winter: ausgeglichener bis leichter Import.

*Alle 1 GIF(s) dieses Typs liegen in `output/charts/experimental/`.*

---
## 8. GIF E — Kombiniert (Zonenfluss + Cross-Border)<a id='gif_e_K_01c'></a>

[↑ Inhaltsverzeichnis](#toc_K_01c)

GIF C + GIF D in einer einzigen Animation. Interne Flüsse + Grenzflüsse simultan.

In [ ]:
print("\n▶ Starte Erstellung: GIF E Tag — Kombiniert (9/10)")
# 🔬 Zonenfluss + Cross-Border kombiniert

def _dyn_e(ax, fi, nf, get_imb_fn, label):
    for zf,zt,_ in ZONE_PAIRS:
        imb_f=get_imb_fn(zf); imb_t=get_imb_fn(zt)
        if imb_f>0 and imb_t<0:   mw=min(abs(imb_f),abs(imb_t))
        elif imb_t>0 and imb_f<0: mw=-min(abs(imb_t),abs(imb_f))
        else:
            mw=imb_f-imb_t
            if abs(mw)<60: continue
        draw_flow_dots(ax,zf,zt,mw,fi,nf,_get_mw_norm(zf,zt,mw,PATH_NORMS))
    total=sum(get_imb_fn(z) for z in ZONE_COLORS_B)
    for nb_key,mw in {nb:total*w for nb,w in NEIGHBOR_WEIGHT.items()}.items():
        mw_n=BORDER_NORM if GIF_C_NORM_MODE=='global' else max(abs(mw),100)
        draw_border_dots(ax,nb_key,mw,fi,nf,mw_n,CH_CENTER)
    for zone,(cx,cy) in ZONE_CENTERS.items():
        imb=get_imb_fn(zone)
        col='#66BB6A' if imb>0 else '#EF5350'
        ax.scatter(cx,cy,s=max(30,min(200,abs(imb)/6)),c=col,alpha=0.45,zorder=7,linewidths=0)
    add_timestamp_bar(ax,"Zonenfluss + Cross-Border",label,"⚠️ Synthetische Daten")

for saison in SAISONS:
    def make_e(s):
        def fn(ax,fi,nf):
            t_h=TAG_TIMES[fi]; h=int(round(t_h))%24
            _dyn_e(ax,fi,nf,lambda z:float(SPLINES_H[s][z]['imb'](t_h)),
                   f"Energiefluss — {h:02d}:00 ({s})")
        return fn
    path=os.path.join(EXP_CHARTS_DIR,f"EXP_kuer_k01c_gif_e_tag_{saison.lower()}.gif")
    make_gif_fast(make_e(saison), N_FRAMES_TAG, FPS_TAG, path, zone_alpha=0.09)


In [ ]:
# ── Ausgabe anzeigen ────────────────────────────────────────────
_p = os.path.join(EXP_CHARTS_DIR, 'EXP_kuer_k01c_gif_e_tag_winter.gif')
# Umschalten: mode='gif' (Standbild) oder mode='slider' (interaktiv).
# framerate nur für Slider relevant (bei GIF steckt fps im GIF selbst).
show_animation(_p, mode='gif', framerate=10)


**GIF E — Kombiniert Tagesverlauf (Winter)**

GIF C + GIF D zusammen. Vollständiges Bild: Interne Flüsse + Grenzexporte/-importe.
Für echte Topologie → K_01d.

*Alle 4 GIF(s) dieses Typs liegen in `output/charts/experimental/`.*

In [ ]:
print("\n▶ Starte Erstellung: GIF E Jahr — Kombiniert (10/10)")
def _dyn_e_j(ax,fi,nf):
    t_w=JAHR_TIMES[fi]; m=MONAT_KURZ[int(t_w/52*12)%12]
    _dyn_e(ax,fi,nf,lambda z:float(SPLINES_W[z]['imb'](t_w)),
           f"Energiefluss — Woche {int(t_w)+1:02d}/52 ({m})")
path=os.path.join(EXP_CHARTS_DIR,'EXP_kuer_k01c_gif_e_jahr.gif')
make_gif_fast(_dyn_e_j, N_FRAMES_JAHR, FPS_JAHR, path, zone_alpha=0.09)


In [ ]:
# ── Ausgabe anzeigen ────────────────────────────────────────────
_p = os.path.join(EXP_CHARTS_DIR, 'EXP_kuer_k01c_gif_e_jahr.gif')
# Umschalten: mode='gif' (Standbild) oder mode='slider' (interaktiv).
# framerate nur für Slider relevant (bei GIF steckt fps im GIF selbst).
show_animation(_p, mode='gif', framerate=10)


**GIF E — Kombiniert Jahresverlauf**

Jahresverlauf der kombinierten Flüsse. Richtungsumkehr sichtbar wenn CH-Saldo negativ.
Für echte Lastflüsse auf realer Topologie: K_01d mit PyPSA-Netz.

*Alle 1 GIF(s) dieses Typs liegen in `output/charts/experimental/`.*

---
## Abschluss<a id='abschluss_K_01c'></a>

[↑ Inhaltsverzeichnis](#toc_K_01c)

In [ ]:
# ── Abschlusskontrolle K_01c ─────────────────────────────────────────────────
print('K_01c – Abschlusskontrolle')
print('=' * 65)

expected = []
for saison in ['winter','frühling','sommer','herbst']:
    for gif in ['a','b','c','d','e']:
        expected.append(f'EXP_kuer_k01c_gif_{gif}_tag_{saison}.gif')
for gif in ['a','b','c','d','e']:
    expected.append(f'EXP_kuer_k01c_gif_{gif}_jahr.gif')

total_size = 0
ok_count = 0
for fname in expected:
    p = os.path.join(EXP_CHARTS_DIR, fname)
    exists = os.path.exists(p)
    size_kb = os.path.getsize(p)//1024 if exists else 0
    icon = '✅' if exists else '❌'
    print(f"  {icon} {fname:<48} {size_kb:>5} KB")
    if exists:
        ok_count += 1
        total_size += size_kb

print()
print(f"  Total: {ok_count}/{len(expected)} GIFs | {total_size//1024:.1f} MB")
print()
print('⚙ Parameter (in diesem NB):')
print(f'  DPI_GIF={DPI_GIF} | FPS_TAG={FPS_TAG} | FPS_JAHR={FPS_JAHR}')
print()
print('Hinweis: Alle Flussdaten sind synthetisch modelliert.')
print('Für operative Analysen: Swissgrid SCADA oder ENTSO-E Transparency API.')


| [← K_01b Zonenmodell](K_01b_Zonenmodell_Erweitert.ipynb) | [↑ Übersicht](../organisation/O_01_Project_Overview.ipynb) | [K_02 Cross-Border →](K_02_Cross_Border.ipynb) |
|:---|:---:|---:|